# 06 — Preparación y controles de SOFA dinámico

Verifica disponibilidad agregada de los seis componentes y demuestra las funciones de puntuación ya probadas. Este notebook **no etiqueta Sepsis-3**: primero debe materializarse el grafo horario oficial de `mimic-code v2.4.0` para MIMIC-IV Demo v2.2.

La puntuación se calcula para cada observación con contexto contemporáneo y después se selecciona el peor score de la ventana móvil de 24 horas.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT / 'src') not in sys.path: sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from mimic_sepsis.sofa import (respiratory_score, coagulation_score, liver_score, cardiovascular_score, cns_score, renal_score, total_sofa)
DATA_DIR = PROJECT_ROOT / 'data' / 'mimic-iv-demo' / '2.2'

## Versión y grafo normativo

- Demo MIMIC-IV v2.2: `mimic-code v2.4.0` (`570ef01`).
- Despliegue MIMIC-IV v3.1: `mimic-code v3.0.1` (`c7e0756`).
- Dependencias y orden: `docs/mimic_code_sofa_plan.md`.
- Itemids y cobertura: `docs/sofa_demo_inventory.md`.

In [ ]:
coverage = pd.DataFrame([
 {'component':'Respiratorio: PaO2','stays':87,'percent':62.1},
 {'component':'Respiratorio: FiO2','stays':76,'percent':54.3},
 {'component':'Ventilación invasiva','stays':60,'percent':42.9},
 {'component':'Coagulación','stays':133,'percent':95.0},
 {'component':'Hepático','stays':65,'percent':46.4},
 {'component':'Cardiovascular: MAP','stays':65,'percent':46.4},
 {'component':'Cardiovascular: vasoactivos','stays':35,'percent':25.0},
 {'component':'Neurológico','stays':140,'percent':100.0},
 {'component':'Renal: creatinina','stays':137,'percent':97.9},
 {'component':'Renal: diuresis','stays':137,'percent':97.9},
])
coverage

Los porcentajes indican presencia alguna vez durante la estancia, no disponibilidad en cada hora o ventana de 24 horas. No deben interpretarse como completitud del SOFA dinámico.

## Casos frontera de puntuación

In [ ]:
boundary_checks = pd.DataFrame([
 {'component':'respiratory','input':'PF=99, invasive','score':respiratory_score(99, True)},
 {'component':'respiratory','input':'PF=99, not invasive','score':respiratory_score(99, False)},
 {'component':'coagulation','input':'platelets=49 K/uL','score':coagulation_score(49)},
 {'component':'liver','input':'bilirubin=6 mg/dL','score':liver_score(6)},
 {'component':'cardiovascular','input':'norepinephrine=0.1 mcg/kg/min','score':cardiovascular_score(norepinephrine=0.1)},
 {'component':'cns','input':'GCS=9','score':cns_score(9)},
 {'component':'renal','input':'creatinine=1; urine=199 mL/24h','score':renal_score(1,199)},
])
boundary_checks

## Control crítico de contemporaneidad respiratoria

In [ ]:
respiratory_observations = pd.DataFrame([
 {'time':'00:00','pf_ratio':68,'invasive_ventilation':False},
 {'time':'06:00','pf_ratio':120,'invasive_ventilation':True},
])
respiratory_observations['score'] = respiratory_observations.apply(lambda row: respiratory_score(row.pf_ratio, row.invasive_ventilation), axis=1)
assert respiratory_observations.score.max() == 3
respiratory_observations

Tomar `min(PF)=68` y combinarlo con `any(ventilación)=True` produciría erróneamente un score 4. El pipeline puntuará primero cada observación con su soporte contemporáneo y luego tomará el máximo de la ventana.

## Missingness explícito

In [ ]:
components = [2, 1, None, 3, 0, 1]
strict_total = total_sofa(*components)
mimic_total = total_sofa(*components, mimic_missing_components_as_zero=True)
pd.DataFrame([{'strategy':'strict_complete_components','sofa':strict_total,'missing_components':1},{'strategy':'mimic_missing_components_as_zero','sofa':mimic_total,'missing_components':1}])

La imputación de componentes ausentes a cero reproduce el concepto MIMIC, pero no es el mismo supuesto que considerar SOFA basal cero en Sepsis-3. Siempre se conservará el número y nombre de componentes ausentes y se realizará una sensibilidad de casos completos.

## Criterio para avanzar al outcome

1. Materializar el subgrafo oficial horario.
2. Validar unidades, rangos, duplicados e intervalos de infusión.
3. Auditar cobertura por componente y hora.
4. Comparar distribuciones con conceptos derivados de la misma versión.
5. Solo entonces enlazar SOFA con sospecha de infección y calcular `t0`.